In [ ]:
from binance.client import Client
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
import psycopg2
import boto3
import io
import ipywidgets as widgets
import time
%matplotlib inline

In [ ]:
def get_secret(secret_name):
    region_name = "eu-west-1"
    # Create a Secrets Manager client
    client = boto3.client('secretsmanager', region_name='eu-west-1')
    get_secret_value_response = client.get_secret_value(SecretId=secret_name)
    if 'SecretString' in get_secret_value_response:
        secret = get_secret_value_response['SecretString']
    else:
        secret = base64.b64decode(get_secret_value_response['SecretBinary'])
    return json.loads(secret)
def get_klines_futures(symbol, start_str):
    data = client.futures_historical_klines(symbol=symbol, interval=client.KLINE_INTERVAL_1HOUR, start_str=start_str)
    ser = pd.Series([vv[4] for vv in data], dtype=float, index=pd.to_datetime([vv[6] for vv in data], unit='ms'))
    ser.name = symbol
    return ser.iloc[:-1]
sec = get_secret('binance/binance')
client = Client('YOUR_BINANCE_API_KEY', 'YOUR_BINANCE_API_SECRET')

In [ ]:
tickers = client.futures_symbol_ticker()
perpetual = []
for i, ts in enumerate(tickers):
    ticker = ts['symbol']
    if ticker.endswith('USDT'):
        print(i, len(tickers))
        perpetual.append(get_klines_futures(ticker, '1-Jan-2021'))
        time.sleep(0.001)
perpetual = pd.concat(perpetual, axis=1)
historical_live_base = perpetual.copy()

In [ ]:
to_keep = ['BTCUSDT',
 'ETHUSDT',
 'BCHUSDT',
 'XRPUSDT',
 'EOSUSDT',
 'LTCUSDT',
 'TRXUSDT',
 'ETCUSDT',
 'XLMUSDT',
 'XMRUSDT',
 'DASHUSDT',
 'ZECUSDT',
 'XTZUSDT',
 'ATOMUSDT',
 'IOTAUSDT',
 'BATUSDT',
 'VETUSDT',
 'NEOUSDT',
 'QTUMUSDT',
 'IOSTUSDT',
 'THETAUSDT',
 'ZILUSDT',
 'OMGUSDT',
 'RLCUSDT',
 'WAVESUSDT',
 'EGLDUSDT',
 'SOLUSDT',
 'ICXUSDT',
 'STORJUSDT',
 'BLZUSDT',
 'AVAXUSDT',
 'FTMUSDT',
 'HNTUSDT',
 'TOMOUSDT',
 'KSMUSDT',
 'NEARUSDT',
 'FILUSDT',
 'MATICUSDT',
 'OCEANUSDT',
 'CVCUSDT',
 'ZENUSDT',
 'SKLUSDT',
 'GRTUSDT',
 'ANKRUSDT',
 'BTSUSDT',
 'LITUSDT',
 'REEFUSDT',
 'RVNUSDT',
 'XEMUSDT',
 'COTIUSDT',
 'HBARUSDT',
 'ONEUSDT',
 'STMXUSDT',
 'DENTUSDT',
 'CELRUSDT',
 'HOTUSDT',
 'MTLUSDT',
 'BTTUSDT',
 'NKNUSDT',
 'SCUSDT',
 'DGBUSDT',
 'ICPUSDT',
 'GTCUSDT']

In [ ]:
perpetual2 = []
start_str = historical_live_base.index[-1] + datetime.timedelta(seconds=1)
start_str = start_str.strftime('%d-%b-%Y %H:%M')
print(start_str)
for i, ticker in enumerate(historical_live_base.columns):
    print(i, len(tickers))
    perpetual2.append(get_klines_futures(ticker, start_str))
perpetual2 = pd.concat(perpetual2, axis=1)
historical_live = pd.concat([historical_live_base, perpetual2], axis=0)
historical_live

In [ ]:
def make_positions_smooth(x, max_active):
    tmp = x.copy()
    sorted_idx = tmp.sort_values(ascending=False).dropna().index
    weights = 0*tmp
    to_short = tmp[tmp<0].dropna().sort_values()[:max_active]
    to_long = tmp[tmp>0].dropna().sort_values()[-max_active:]
    weights.loc[to_short.index] = -1/max_active/2
    weights.loc[to_long.index] = 1/max_active/2
    #weights.loc['DEFIUSDT'] = -0.2
    return weights
def simulate(df, wsize=None, wsize_short=None, max_active=None):
    returns = df.diff(1)/df.shift(1)
    returns_adj = returns#/returns.ewm(com=30).std()
    signal = 0.
    if wsize is not None:
        signal += returns_adj.ewm(com=wsize).mean()
    if wsize_short is not None:
        signal -= returns_adj.ewm(com=wsize_short).mean()
    signal[signal.abs() < 1e-10] = 0.
    if False:
        signal_bef = signal.copy()
        for i in range(len(signal)):
            np.random.shuffle(signal.iloc[i, :])
        print((signal_bef-signal).sum().sum())
    positions = signal.apply(lambda x: make_positions_smooth(x, max_active), axis=1)
    if False:
        positions_holding = positions.iloc[::3, :].copy()
        positions_holding = positions_holding.reindex(positions.index).ffill()
    else:
        positions_holding = positions
    pnl = (positions_holding.shift(1)*returns)
    trades = positions_holding.diff(1).abs()
    cost = trades*0.0004
    return pnl, positions_holding, cost, signal
def get_prev_date(curr_date, days_before, hours_before):
    res = curr_date - datetime.timedelta(days=days_before, hours=hours_before)
    return res
def plot_dashboard(pnl, pos, cost, signal, start_date=None, do_plot=True):
    if start_date:
        _pnl = pnl.loc[start_date:, :]
        _pos = pos.loc[start_date:, :]
        _cost = cost.loc[start_date:, :]
        _signal = signal.loc[start_date:, :]
    else:
        _pnl = pnl
        _pos = pos
        _cost = cost
        _signal = signal
    (_pnl).sum(axis=1).cumsum().plot()
    tmp = (_pnl - _cost).sum(axis=1)
    tmp.cumsum().plot()
    scores = (tmp.sum(), tmp.mean()/tmp.std(), _cost.sum(axis=1).sum())
    print(scores)
    if do_plot:
        plt.figure()
        monthly = tmp.resample('M').sum()
        monthly.plot(kind='bar', figsize=(10, 5))
        plt.ylabel('Pnl (monthly)')
        print(monthly.iloc[-1])
        plt.figure()
        (_pos>0).sum(axis=1).resample('W').mean().plot()
        plt.ylabel('Num Active (weekly avg)')
        plt.figure()
        weekly = _cost.sum(axis=1).resample('W').sum()
        weekly.plot()
        plt.ylabel('Cost (weekly total)')
    return scores

In [ ]:
calc_start = datetime.datetime.strptime('1-Feb-2021', '%d-%b-%Y')
print(calc_start)
score_params = []
for wsize in [5, 10, 35, 55, 65, 95, 125]:
    for wsize_short in [5, 10, 25, 35, 55]:
        if wsize <= wsize_short:
            continue
        for p1 in [50, 10, 5, 2]:
            print(wsize, wsize_short, p1)
            pstart = calc_start - datetime.timedelta(days=15)
            prices = historical_live.loc[pstart.strftime('%d-%b-%Y'):, to_keep]
            pnl_global, pos_global, cost_global, signal_global = simulate(prices, wsize, wsize_short, p1)
            score = plot_dashboard(pnl_global, pos_global, cost_global, signal_global, calc_start.strftime('%d-%b-%Y %H:%M'), False)[0]
            score_params.append((score, (wsize, wsize_short, p1)))
score_params = sorted(score_params, key=lambda x: x[0])
print(score_params[-5:])

In [ ]:

p2 = 0.
wsize = None
wsize_short = 80

p1 = 4# best_p1
start_date = '26-Jun-2021'
prices = historical_live.loc[:, to_keep]
pnl_global, pos_global, cost_global, signal_global = simulate(prices, wsize, wsize_short, p1)
plot_dashboard(pnl_global, pos_global, cost_global, signal_global, start_date, False)


In [ ]:
symbols = pos_global.iloc[-1, :]
symbols = symbols[symbols!=0]
print(symbols)
xx=pnl_global.iloc[-1, :].loc[symbols.index]
print('pnl', xx.sort_values(), xx.sum())
sig = signal_global.iloc[-1, :].loc[symbols.index]
sig.sort_values()

In [ ]:
leverage = 10
fund_size = 300*leverage
for sym, ls in symbols.iteritems():
    pr = float(client.futures_ticker(symbol=sym)['lastPrice'])
    _prec = prec[sym]
    qty = round(fund_size/pr/len(symbols), _prec)
    side = 'SELL' if ls<0 else 'BUY'
    print(sym, qty, side)
    
    #client.futures_create_order(symbol=sym, side=side, type='MARKET', quantity=qty)

In [ ]:
client.futures_create_order()

In [ ]:
aa=client.futures_position_information(symbol='DEFIUSDT')
print(float(aa[0]['unRealizedProfit']))
float(aa[0]['unRealizedProfit'])/float(aa[0]['notional'])*100

In [ ]:
client.futures_position_information(symbol='GTCUSDT')

In [ ]:
prec = [x for x in client.futures_exchange_info()['symbols'] if x['contractType'] == 'PERPETUAL' and x['quoteAsset'] == 'USDT']
prec = {x['symbol']: x['quantityPrecision'] for x in prec}
prec

In [ ]:
'DEFIUSDT' in historical_live.columns

In [ ]:
simstart = pnl_global.index[-1].replace(day=1).strftime('%d-%b-%Y')
aa=(pnl_global-cost_global).loc[simstart:, :].sum(axis=1).resample('D').sum()
mkt_ret = historical_live.loc[:, to_keep]
mkt_ret = mkt_ret.diff(1)/mkt_ret.shift(1)
mkt_ret = mkt_ret.loc[simstart:, :]
mkt_ret = mkt_ret.mean(axis=1)
pd.concat([mkt_ret.resample('D').sum(), aa], keys=['Market', 'Trading'], axis=1).plot(kind='bar', figsize=(15,5))
plt.legend(loc='upper left')
plt.ylim((-0.3, 0.5))
plt.grid(True)
plt.figure()
top_k = (pnl_global-cost_global).loc[simstart:, :].sum(axis=0).sort_values(ascending=False)
pd.concat([top_k.iloc[:10], top_k.iloc[-10:]], axis=1, keys=['winners', 'losers']).plot(kind='bar', figsize=(10,5))
plt.figure()
(pnl_global-cost_global).loc[simstart:, :].sum(axis=1).cumsum().plot(marker='.')
mkt_ret.loc[simstart:].cumsum().plot(color='k', marker='.')
aa.sum()

In [ ]:
start_of_day = pnl_global.index[-1].strftime('%d-%b-%Y')
aa=(pnl_global-cost_global).loc[start_of_day:, :]
mkt_ret = historical_live.loc[:, to_keep]
mkt_ret = mkt_ret.diff(1)/mkt_ret.shift(1)
mkt_ret = mkt_ret.loc[start_of_day:, :]
plt.figure()
pd.concat([mkt_ret.mean(axis=1), aa.sum(axis=1)], keys=['Market', 'Trading'], axis=1).plot(kind='bar', figsize=(15,5))
plt.ylim((-.05, 0.075))
plt.grid(True)
plt.legend(loc='lower left')
plt.figure()
top_k = aa.sum(axis=0).sort_values(ascending=False)
pd.concat([top_k.iloc[:10], top_k.iloc[-5:]], axis=1, keys=['winners', 'losers']).plot(kind='bar', figsize=(10,5))
plt.figure()
aa.sum(axis=1).cumsum().plot(marker='.')
mkt_ret.mean(axis=1).cumsum().plot(color='k', marker='.')
aa.sum().sum()

In [ ]:
xx=signal_global.iloc[-1, :]
xx[xx>0]
aa=client.futures_exchange_info()

In [ ]:
ret = historical_live.loc['1-May-2021':, to_keep]
ret = ret.diff(1)/ret.shift(1)
corr = ret.corr()
import matplotlib.pyplot as plt

plt.matshow(corr)
plt.colorbar()
corr[corr==1] = np.nan
maxv1 = corr.max(axis=1).sort_values(ascending=False)
print(maxv1.iloc[:10])
corr[maxv1.index[0]].sort_values(ascending=False)

In [ ]:
perpetual2 = []
start_str = historical_live_base.index[-1] + datetime.timedelta(seconds=1)
start_str = start_str.strftime('%d-%b-%Y %H:%M')
print(start_str)
for i, ticker in enumerate(historical_live_base.columns):
    print(i, len(tickers))
    perpetual2.append(get_klines_futures(ticker, start_str))
perpetual2 = pd.concat(perpetual2, axis=1)
historical_live = pd.concat([historical_live_base, perpetual2], axis=0)#.iloc[:-1, :]
historical_live

In [ ]:
from statsmodels.tsa.stattools import coint
import numpy as np
def eval_pair(data, i, j, reps):
    tmp = data.iloc[:, [i, j]]
    idx = tmp.notna().all(axis=1)
    if idx.sum() < 0.9*len(tmp):
        return
    tmp = tmp.loc[idx]
    curr_l = []
    for _ in range(reps):
        sampled = tmp.sample(frac=0.95)
        if sampled.shape[0] > 30:
            curr_l.append(coint(sampled.iloc[:, 0], sampled.iloc[:, 1])[1])
    if not curr_l:
        return
    curr = np.median(curr_l)
    return (curr, (data.columns[i], data.columns[j])) if curr < 0.001 else None
def find_cointegrated_pairs(data, reps):
    n = data.shape[1]
    pairs = [eval_pair(data, x[0], x[1], reps) for x in [(i, j) for i in range(n) for j in range(i+1, n)]]
    pairs = [x for x in pairs if x]
    return sorted(pairs, key=lambda x: x[0])

In [ ]:
dt = (datetime.datetime.now()-datetime.timedelta(days=5)).strftime('%d-%b-%Y %H:%M')
dt2 = (datetime.datetime.now()-datetime.timedelta(days=0)).strftime('%d-%b-%Y %H:%M')
dt22 = (datetime.datetime.strptime(dt2, '%d-%b-%Y %H:%M')+datetime.timedelta(days=1)).strftime('%d-%b-%Y')
print(dt, dt2)
res = find_cointegrated_pairs(historical_live.loc[dt:dt2, historical_live.columns[:20]])
res

In [ ]:
def calc_zratio(_v1, _v2):
    wsize = 30
    ratio = _v1/_v2
    zratio = (ratio-ratio.rolling(wsize).mean())/ratio.rolling(wsize).std()
    return zratio
def calc_pairs_pnl(_v1, _v2, thr, start_date=None, extra_info=False):
    if start_date is not None:
        dstart = datetime.datetime.strptime(start_date, '%d-%b-%Y')-datetime.timedelta(hours=wsize+10)
        v1 = _v1.loc[dstart:]
        v2 = _v2.loc[dstart:]
    else:
        v1 = _v1
        v2 = _v2
    zratio = calc_zratio(v1, v2)
    ret1 = v1.diff(1)/v1.shift(1)
    ret2 = v2.diff(1)/v2.shift(1)
    pos1 = 0 *zratio
    pos1.loc[(zratio>thr)] = -0.5
    pos1.loc[zratio<-thr] = 0.5
    pos2 = -pos1
    pnl1 = pos1.shift(1)*ret1#*(1-0.002)/(1+0.002)
    pnl2 = pos2.shift(1)*ret2#*(1-0.002)/(1+0.002)
    cost1 = pos1.diff(1).abs().replace(np.nan, 0)*0.0004
    #cost1 += (cost1>0).astype(float)*(1-0.002)/(1+0.002)
    cost2 = pos2.diff(1).abs().replace(np.nan, 0)*0.0004
    #cost2 += (cost2>0).astype(float)*(1-0.002)/(1+0.002)
    net_pnl = pnl1+pnl2-cost1-cost2
    if start_date is not None:
        return (net_pnl.loc[start_date:], (pos1.loc[start_date:], pos2.loc[start_date:]), zratio.loc[start_date:]) if extra_info else net_pnl.loc[start_date:].sum()
    else:
        return (net_pnl, (pos1, pos2), zratio) if extra_info else net_pnl.sum()
def top_pairs_strategy(best_res, start_date, end_date, top_k, thr):
    found = set()
    pfl_idx = []
    for i in range(len(best_res)):
        _, pair = best_res[i]
        if pair[0] in found or pair[1] in found:
            continue
        pfl_idx.append(i)
        if len(pfl_idx) == top_k:
            break
        found.add(pair[0])
        found.add(pair[1])
    pairs = [best_res[i][1] for i in pfl_idx]
    pnl_pfl = []
    for pair in pairs:
        pnl, _, _ = calc_pairs_pnl(historical_live.loc[:end_date, pair[0]], historical_live.loc[:end_date, pair[1]], thr, start_date, True)
        pnl.name='_'.join(list(pair))
        pnl_pfl.append(pnl)
    pnl_pfl = pd.concat(pnl_pfl, axis=1).dropna()
    return pnl_pfl/len(pairs), pairs
def sharpe(pp):
    ser = pp.copy()
    ser[ser>ser.quantile(0.98)] = np.nan
    ser[ser<ser.quantile(0.02)] = np.nan
    return ser.mean()/ser.std()

In [ ]:
best_res =  [['MANAUSDT', 'LITUSDT'], ['ZRXUSDT', 'CELRUSDT'], ['TRXUSDT', 'NKNUSDT']]
best_res = [[None, x] for x in best_res]
sim_start = '15-Jun-2021'
pnl_mem, pairs = top_pairs_strategy(best_res, sim_start, '1-Dec-2025', 3, 0)
print(json.dumps(pairs))
pnl_pfl = pnl_mem.sum(axis=1)
pnl_mem.cumsum().plot()
pnl_pfl.cumsum().plot(color='k')
plt.grid(True)
plt.figure()
pnl_pfl.plot()
pnl_pfl.cumsum().plot(secondary_y=True)
plt.figure()
pnl_pfl.iloc[-80:].plot(kind='bar', figsize=(20,5))
plt.title(pnl_pfl.sum())
plt.figure()
pnl_pfl.resample('1D').sum().iloc[-80:].plot(kind='bar', figsize=(15,5))

In [ ]:
historical_live.to_pickle('cloud/python/server/historical_live.pkl', protocol=4)

In [ ]:
bt_date = datetime.datetime.strptime('30-Jun-2021', '%d-%b-%Y')
bt_step = 5
bt_result = []
while True:
    print(bt_date)
    bt_prev_start = (bt_date-datetime.timedelta(days=5)).strftime('%d-%b-%Y')
    bt_prev_end = (bt_date-datetime.timedelta(days=1)).strftime('%d-%b-%Y')
    bt_next_end = bt_date+datetime.timedelta(days=bt_step-1)
    if bt_date > historical_live.index[-1]:
        break
    bt_next_end = bt_next_end.strftime('%d-%b-%Y')
    #bt_coint = find_cointegrated_pairs(historical_live.loc[bt_prev_start:bt_prev_end, :], 5)
    coint_key = bt_date.strftime('%d-%b-%Y')
    if coint_key not in recent_coints:
        print(coint_key, 'not in recent coints')
    else:
        bt_coint = recent_coints[coint_key]
        bt_coint = [x for x in bt_coint if x[0]<1e-3]
    bt_pnl = []
    for i in range(len(bt_coint)):
        pp, _, _ = calc_pairs_pnl(
                    historical_live.loc[:bt_prev_end, bt_coint[i][1][0]],
            historical_live.loc[:bt_prev_end, bt_coint[i][1][1]],
            0.,
            bt_prev_start, 
            True)
        pp[pp.abs()>0.02] = np.nan
        bt_pnl.append(sharpe(pp))
    bt_pnl = sorted(zip(bt_pnl, bt_coint), key=lambda x: x[0], reverse=True)
    _, bt_pnl = zip(*bt_pnl)
    bt_pnl_mem, _ = top_pairs_strategy(bt_pnl, bt_date.strftime('%d-%b-%Y'), bt_next_end, 3, 0.)
    print(bt_pnl_mem.index[0], bt_pnl_mem.index[-1])
    bt_result.append(bt_pnl_mem)
    bt_date = datetime.datetime.strptime(bt_next_end, '%d-%b-%Y') + datetime.timedelta(days=1)
bt_pfl_pnl = pd.concat([x.sum(axis=1) for x in bt_result], axis=0)

In [ ]:
bt_pfl_pnl.cumsum().plot()
print(bt_pfl_pnl.mean()/bt_pfl_pnl.std(), bt_pfl_pnl.sum())

In [ ]:
xx = bt_pfl_pnl.loc['8-Jul-2021':]
xx.cumsum().plot()
print(xx.mean()/xx.std(), xx.sum())
print(xx.sum()*500*20)
plt.grid(True)

In [ ]:
xx.iloc[-50:].cumsum().plot()

In [ ]:
sd = '1-Jan-2021'
pairs = bt_pnl
zratios = []
cols = sorted(historical_live.columns)
for s1 in cols:
    for s2 in cols:
        if s1==s2:
            continue
        tmp_start = datetime.datetime.strptime(sd, '%d-%b-%Y')-datetime.timedelta(hours=60)
        tmp = calc_zratio(historical_live.loc[tmp_start:, s1], historical_live.loc[tmp_start:, s2])
        tmp.name = f'{s1}_{s2}'
        zratios.append(tmp.loc[sd:])
zratios = pd.concat(zratios, axis=1)
zratios

In [ ]:
recent_coints = {}
idx = datetime.datetime.strptime('1-Jan-2021', '%d-%b-%Y')
prev_coint = set()
while True:
    print(idx)
    rc_start = idx-datetime.timedelta(days=5)
    rc_end = idx-datetime.timedelta(days=1)
    recent_coint = find_cointegrated_pairs(historical_live.loc[rc_start:rc_end, cols], 5)
    print(len(recent_coint))
    recent_coints[idx.strftime('%d-%b-%Y')] = recent_coint
    recent_coint_set = set([x[1] for x in recent_coint])
    print('removals', len(prev_coint-recent_coint_set))
    print('additions', len(recent_coint_set-prev_coint))
    prev_coint = recent_coint_set
    idx += datetime.timedelta(days=5)
    if idx > historical_live.index[-1]:
        break

In [ ]:
pickle.dump(recent_coints, open('recent_coints_5d.pkl', 'wb'))

In [ ]:
holdings = []
num_pairs = 6
money = []
holdings_series = []
test_input = zratios.loc['1-Jan-2021':, :]
recent_coint = [x[1] for x in recent_coints['01-Jan-2021']]
for i, (idx, row) in enumerate(test_input.iterrows()):
    new_holdings = []
    curr_money = 0
    for holding in holdings:
        pair = holding['pair']
        if holding['zratio']*row['_'.join(pair)] < 0:
            pinit1, pinit2 = holding['prices']
            pfinal1 = historical_live.loc[idx, pair[0]]
            pfinal2 = historical_live.loc[idx, pair[1]]
            r1 = (pfinal1-pinit1)/pinit1
            r2 = (pfinal2-pinit2)/pinit2
            curr_money += (r1-r2)*(1 if holding['zratio']<0 else -1)/num_pairs/2.
        else:
            new_holdings.append(holding)
    money.append(curr_money)
    if idx.strftime('%d-%b-%Y') in recent_coints:
        recent_coint = list(recent_coints[idx.strftime('%d-%b-%Y')])
        recent_coint = [x[1] for x in recent_coint if x[0] < 1e-10]
    to_collect = num_pairs-len(new_holdings)
    if to_collect:
        max_idx = row.loc[['_'.join(x) for x in recent_coint]].dropna().abs().sort_values(ascending=False).index
        new_pairs = row.loc[max_idx]
        curr_pairs = set([x['pair'] for x in new_holdings])
        for pair, ratio in new_pairs.items():
            m1, m2 = pair.split('_')
            if (m1, m2) in curr_pairs or abs(ratio) < 0:
                continue
            hold = {'zratio': ratio, 'pair': (m1, m2), 'prices': (historical_live.loc[idx, m1], historical_live.loc[idx, m2])}
            new_holdings.append(hold)
            if len(new_holdings) == num_pairs:
                break
    holdings = new_holdings
    holdings_series.append(len(holdings))
money = pd.Series(money, index=test_input.index)
money.cumsum().plot()
holdings_series = pd.Series(holdings_series, index=test_input.index)
plt.figure()
holdings_series.plot()
print(money.mean()/money.std())

In [ ]:
st = '1-Mar-2021'
money.loc[st:].cumsum().plot()
money.loc[st:].mean()/money.loc[st:].std()

In [ ]:
#pd.read_pickle('diversified_1_jan.pkl').sum(axis=1).cumsum().plot(label='diversified_1_jan')
aa=pd.read_pickle('5_0_0_30_sr_3_mar_bootstrap_latest.pkl').loc['1-Mar-2021':]
aa.cumsum().plot(label='5_0_0_30_sr_3_mar_bootstrap_latest')

plt.legend()
print(aa.mean()/aa.std())

In [ ]:
st = '1-Jun-2021'
with open('5_0_0_30_sr_3_mar_bootstrap.pkl', 'rb') as f:
    btr = pickle.load(f)
    tplt = pd.concat([x.sum(axis=1) for x in btr], axis=0)
    tplt.loc[st:].cumsum().plot(label=fl)
    print(fl, sharpe(tplt))
tplt = pd.read_pickle('5_0_0_30_sr_3_mar_bootstrap_latest.pkl')
tplt.loc[st:].cumsum().plot(label='5_0_0_30_sr_3_mar_bootstrap_latest')
pd.read_pickle('diversified_1_jan.pkl').sum(axis=1).loc[st:].cumsum().plot(label='diversified_1_jan')
plt.legend()

In [ ]:
with open('5_0_0_30_sr_3_jan_bootstrap.pkl', 'rb') as f:
    btr = pickle.load(f)
    tplt1 = pd.concat([x.sum(axis=1) for x in btr], axis=0)
    tplt1.loc['Mar-2021':].cumsum().plot(label=fl)
    print(fl, sharpe(tplt1))
with open('5_0_0_30_sr_3_mar_bootstrap.pkl', 'rb') as f:
    btr = pickle.load(f)
    tplt2 = pd.concat([x.sum(axis=1) for x in btr], axis=0)
    tplt2.cumsum().plot(label=fl)
    print(fl, sharpe(tplt2))
plt.figure()
(tplt1.cumsum()-tplt2.cumsum()).plot()

In [ ]:
with open('5_0_0_30_sr_3_jan_bootstrap.pkl', 'rb') as f:
    btr = pickle.load(f)
    tplt1 = pd.concat([x.sum(axis=1) for x in btr], axis=0)
    tplt1.cumsum().plot(label=fl)
    print(fl, sharpe(tplt1))

In [ ]:
with open('5_0_0_30_sr_3_mar_bootstrap.pkl', 'rb') as f:
    btr = pickle.load(f)
    p1 = pd.concat([x.sum(axis=1) for x in btr], axis=0)
p1.cumsum().plot()
print(sharpe(p1))
plt.figure()
p1.iloc[-50:].plot(kind='bar', figsize=(15,5))
plt.figure()
p1.resample('1D').sum().iloc[-50:].plot(kind='bar', figsize=(15,5))

In [ ]:
sd = p1.index[0].date()
acc = []
for i in range(5):
    acc.append(p1.loc[sd+datetime.timedelta(days=i):].resample('1D').sum().iloc[::5].reset_index(drop=True))
acc = pd.concat(acc, axis=1)
acc.sum(axis=0).plot(kind='bar', yerr=acc.std(axis=0))

In [ ]:
acc.cumsum(axis=0).plot()

In [ ]:
def calc_ratio(v1, v2):
    ratio = v1/v2
    zratio = (ratio-ratio.rolling(wsize).mean())/ratio.rolling(wsize).std()
    return zratio
def pairs_pos(s):
    tmp = s.replace(np.nan, 0)
    buy = tmp[tmp>0].sort_values(ascending=False).iloc[:5]
    sell = tmp[tmp<0].sort_values().iloc[:5]
    res = tmp*0
    res[buy.index] = 1
    res[sell.index] = -1
    return res/(len(buy)+len(sell))
bt_date = datetime.datetime.strptime('1-Mar-2021', '%d-%b-%Y')
bt_step = 15
bt_result = []
wsize = 30
prev_coint = set()
while True:
    print(bt_date)
    bt_prev_start = (bt_date-datetime.timedelta(days=bt_step)).strftime('%d-%b-%Y')
    bt_prev_end = (bt_date-datetime.timedelta(days=1)).strftime('%d-%b-%Y')
    bt_next_end = bt_date+datetime.timedelta(days=bt_step-1)
    if bt_date > historical_live.index[-1]:
        break
    bt_next_end = bt_next_end.strftime('%d-%b-%Y')
    bt_coint = find_cointegrated_pairs(historical_live.loc[bt_prev_start:bt_prev_end, :], 5)
    bt_pnl = []
    tmp_price = historical_live.loc[bt_date-datetime.timedelta(hours=wsize*2):bt_next_end, :]
    signal = tmp_price*0.
    signal = signal.replace(np.nan, 0)
    coint_topk = [x for x in bt_coint if x[0] < 1e-10]
    new_coint = set()
    for pval, (m1, m2) in coint_topk:
        new_coint.add(m1)
        new_coint.add(m2)
        r = calc_ratio(tmp_price[m1], tmp_price[m2])
        signal[m1] -= r
        signal[m2] += r
    print('added', len(new_coint-prev_coint), len(prev_coint-new_coint), len(new_coint))
    prev_coint = new_coint
    counts = pd.Series(Counter([x for _, p in coint_topk for x in p]))
    signal = signal/counts
    pos = signal.apply(pairs_pos, axis=1)
    ret = tmp_price.diff(1)/tmp_price.shift(1)
    pnl = (pos.shift(1)*ret).loc[bt_date:, :]
    bt_result.append(pnl)
    bt_date = datetime.datetime.strptime(bt_next_end, '%d-%b-%Y') + datetime.timedelta(days=1)
bt_pnl = pd.concat(bt_result, axis=0)

In [ ]:
bt_pnl_5 = bt_pnl

In [ ]:
bt_pnl.sum(axis=1).cumsum().plot()
bt_pnl_15.sum(axis=1).cumsum().plot()
bt_pnl_5.sum(axis=1).cumsum().plot()
sharpe(bt_pnl_15.sum(axis=1))

In [ ]:
aa=find_cointegrated_pairs(historical_live.iloc[-15*24:, :], 5)
aa

In [ ]:
with open('universe_diversified.json', 'w') as f:
    json.dump([x[1] for x in aa], f)

In [ ]:
tmp_price = historical_live
signal = tmp_price*0.
signal = signal.replace(np.nan, 0)
for pval, (m1, m2) in aa:
    r = calc_ratio(tmp_price[m1], tmp_price[m2])
    signal[m1] -= r
    signal[m2] += r
counts = pd.Series(Counter([x for _, p in aa for x in p]))
signal = signal/counts
signal

In [ ]:
bb=signal.iloc[-1, :]
bb[bb!=0].dropna().sort_values()

In [ ]:
info = client.futures_position_information()
info = [x for x in info if float(x['positionAmt']) != 0]
sum([float(x['unRealizedProfit']) for x in info if x['symbol'] not in ['DEFIUSDT', 'DOGEUSDT']])

In [ ]:
((pd.Series([1,2,-4])>0)*2-1)

In [ ]:
pnl.iloc[-20:]

In [ ]:
start_date = datetime.datetime.now() - datetime.timedelta(hours=10)
start_date = start_date.strftime('%d-%b-%Y %H:%M')
with psycopg2.connect('host=localhost user=postgres password=YOUR_DB_PASSWORD port=5432') as conn:
    with conn.cursor() as cur:
        cur.execute(f"select time, symbol, balance from account where account_name='account1' and time >= '{start_date}'")
        data = cur.fetchall()
conn.close()
balance_split = pd.DataFrame(data, columns=['time', 'symbol', 'balance']).set_index('time')[['balance', 'symbol']]
balance_split = balance_split.pivot(columns='symbol').droplevel(0, axis=1).fillna(0.)
balance = balance_split.sum(axis=1)
balance.plot(marker='o')

In [ ]:
balance_ret = (balance.diff(1)/balance.shift(1)).shift(-1)
sim_ret = (pnl_global-cost_global).sum(axis=1).reindex(balance.index)
pd.concat([balance_ret, sim_ret], axis=1, keys=['real', 'sim']).plot(kind='bar')
plt.figure()
(balance_ret-sim_ret).plot()

In [ ]:
sim_balance = sim_ret.cumsum()*5000
sim_balance += balance.iloc[0]
sim_balance.plot(label='sim', marker='o')
balance.shift(-1).plot(label='real', marker='o')
plt.legend()
(balance.iloc[-1]-sim_balance.iloc[-1])/balance.iloc[-1]

In [ ]:
hr = 20
pp=(pos_global).loc[f'14-Jun-2021 {hr}:00']
pp = pp[pp.abs()>0]
print(pp)
xx=(pnl_global-cost_global).loc[f'14-Jun-2021 {hr+1}:00', pp.index]
print(xx)
xx=balance_split.loc[f'14-Jun-2021 {hr+1}:00', [c for c in to_keep_global if 'DOWNUSDT' not in c]]
print(xx[xx>0])
xx=balance_split.loc[f'14-Jun-2021 {hr+2}:00', [c for c in to_keep_global if 'DOWNUSDT' not in c]]
xx[xx>0]

In [ ]:
print(historical_live.loc[f'14-Jun-2021 {hr}:00', pp.index])
historical_live.loc[f'14-Jun-2021 {hr+1}:00', pp.index]

In [ ]:
aa=balance_split.diff(1)/balance_split.shift(1)
aa = aa.replace(float('inf'), np.nan)
aa.max(axis=1)

In [ ]:
def dow(dd, bf):
    start = '1-Jan-2000'
    tmp_pnl = pnl_global.loc[start:, :]
    if bf:
        tmp = tmp_pnl.loc[[x for x in tmp_pnl.index if x.day_name() == dd and x.hour < 15]].sum(axis=1)
    else:
        tmp = tmp_pnl.loc[[x for x in tmp_pnl.index if x.day_name() == dd and x.hour >= 15]].sum(axis=1)
    tmp[tmp > tmp.quantile(0.98)] = np.nan
    tmp[tmp < tmp.quantile(0.02)] = np.nan
    plt.figure()
    tmp.hist(bins=40)
    #tmp.plot()
    plt.title(dd)
    return tmp.mean()*100
print(dow('Saturday', True), dow('Saturday', False))
print(dow('Sunday', True), dow('Sunday', False))
print(dow('Monday', True), dow('Monday', False))
print(dow('Tuesday', True), dow('Tuesday', False))
print(dow('Wednesday', True), dow('Wednesday', False))
print(dow('Thursday', True), dow('Thursday', False))
print(dow('Friday', True), dow('Friday', False))

pnl_global.sum(axis=1).mean()

In [ ]:
start_date = datetime.datetime.now() - datetime.timedelta(hours=6)
start_date = start_date.strftime('%d-%b-%Y %H:%M')
with psycopg2.connect('host=localhost user=postgres password=YOUR_DB_PASSWORD port=5432') as conn:
    conn.autocommit = True
    with conn.cursor() as cur:
        qry = f"select time, symbol, (fill_price-target_price)/target_price*100, side from slippage where time >= '{start_date}'"
        cur.execute(qry)
        data = cur.fetchall()
conn.close()
slippage = pd.DataFrame(data, columns=['time', 'symbol', 'slippage', 'side'])
slippage

In [ ]:
tmp = slippage.copy()
tmp['date'] = tmp['time'].apply(lambda x: x.date())
grp = tmp.loc[tmp['side']=='BUY', :].groupby('date').mean()
print(grp.iloc[-3:])
grp.plot(kind='bar', figsize=(15,5))
grp = tmp.loc[tmp['side']=='SELL', :].groupby('date').mean()
grp.plot(ax=plt.gca(), kind='bar', color='g')
plt.axhline(0, color='k')
print(grp.iloc[-3:])

In [ ]:
if False:
    #tmp = slippage.loc[[x in high_vol.index for x in slippage['symbol']], :].copy()
    # tmp = slippage.loc[[x not in truncate_slippage for x in slippage['symbol']], :].copy()
    tmp = slippage.loc[[x not in to_keep_trunc for x in slippage['symbol']], :].copy()
else:
    tmp = slippage.copy()
aa=tmp.loc[slippage['side'] == 'SELL', 'slippage']
aa.hist(bins=40)
plt.title('SELL')
print('SELL', aa.mean())
plt.figure()
aa=tmp.loc[slippage['side'] == 'BUY', 'slippage']
aa.hist(bins=40)
plt.title('BUY')
print('BUY', aa.mean())

In [ ]:
worst_sell=slippage.loc[slippage['side'] == 'SELL', ['symbol', 'slippage']].groupby('symbol')
worst_sell=worst_sell.mean().sort_values('slippage')
worst_sell.plot(kind='bar', figsize=(15, 5))
plt.title('SELL')
plt.figure()
worst_buy=slippage.loc[slippage['side'] == 'BUY', ['symbol', 'slippage']].groupby('symbol')
worst_buy=worst_buy.mean().sort_values('slippage')
worst_buy.plot(kind='bar', figsize=(15,5))
plt.title('BUY')

In [ ]:
start_date = datetime.datetime.now() - datetime.timedelta(hours=24*5)
start_date = start_date.strftime('%d-%b-%Y %H:%M')
with psycopg2.connect('host=localhost user=postgres password=YOUR_DB_PASSWORD port=5432') as conn:
    with conn.cursor() as cur:
        cur.execute(f"select time, symbol, bid, ask from orderbook where time >= '{start_date}'")
        data = cur.fetchall()
conn.close()
spreads = pd.DataFrame(data, columns=['time', 'symbol', 'bid', 'ask'])
spreads['spread'] = (spreads['ask']-spreads['bid'])/spreads['ask']*100
spreads = spreads.drop(['bid', 'ask'], axis=1)
spreads = spreads.pivot(columns='symbol', index='time').droplevel(0, axis=1)
spreads = spreads.loc[:, to_keep_global]
spreads = spreads.loc[: ,[x for x in spreads.columns if 'DOWN' not in x]]
spreads

In [ ]:
toplt = spreads.mean(axis=0).sort_values(ascending=False).index
spreads.loc[:, toplt].mean(axis=0).plot(kind='bar', figsize=(15,5))
plt.figure()
spreads.mean(axis=0).sort_values(ascending=False).plot()

In [ ]:
traded = set(to_keep_global)
sp = spreads.loc[:, [x for x in spreads.columns if x in traded]].mean(axis=0).sort_values(ascending=False).dropna()
truncate_slippage = sp.loc[sp>0.2].index
print(truncate_slippage)
entry = set(truncate_slippage) - set(to_keep_trunc)
print('Entry', entry)
print(sp.loc[entry])
exit = set(to_keep_trunc) - set(truncate_slippage)
print('Exit', exit)
sp.loc[exit]

In [ ]:
spreads.loc[:, 'COSUSDT'].plot()

In [ ]:
plt.figure()
axs = plt.gca()
spreads.loc[:, list(truncate_slippage)].rolling(15).mean().plot(color='gray', ax=axs)
plt.ylim((0, 0.7))
plt.legend([])
plt.figure()
spreads.loc[:, [x for x in sp.index if x not in truncate_slippage]].rolling(15).mean().plot(color='gray')
plt.ylim((0, 0.7))
plt.legend([])

In [ ]:
spreads.loc[:, 'KEYUSDT'].rolling(5).mean().plot()
spreads.loc[:, 'DATAUSDT'].rolling(5).mean().plot()

In [ ]:
sp

In [ ]:
from statsmodels.tsa.stattools import pacf
def get_adjusted_pcaf(x, nlags):
    val, cint = pacf(x.dropna(), nlags=nlags, alpha=0.05)
    adj = val[1:]/(cint[1:,1]-cint[1:,0])*2
    return adj
cdata = []
cols =  ['DEFIUSDT']
start_date = '1-Jan-2021'
for c in cols:
    aa=historical_live.loc[start_date:, c]
    ret = aa.diff(1)/aa.shift(1)
    ret = ret/ret.std()
    ret.loc[ret.abs() > 5] = np.nan
    ret = ret.dropna()
    nlags = 10
    if len(ret) < nlags*2:
        continue
    ser = pd.Series(get_adjusted_pcaf(ret, nlags), index=list(range(1, nlags+1)))
    ser.name = c
    cdata.append(ser)
cdata = pd.concat(cdata, axis=1)
plt.axhspan(-1, 1, alpha=0.5, color='y')
cdata.mean(axis=1).plot(marker='o')


plt.grid(True)
#plt.ylim((-0.2, 0.2))
print(cdata.loc[[1,2,3,4,5], :].mean(axis=1).mean())

In [ ]:
infos

In [ ]:
with psycopg2.connect('host=localhost user=postgres password=YOUR_DB_PASSWORD port=5432') as conn:
    conn.autocommit = True
    with conn.cursor() as cur:
        for info in infos:
            symbol = info['symbol']
            print(symbol)
            for f in info['filters']:
                ftype = f['filterType']
                fval = dict(f)
                del fval['filterType']
                qry = 'insert into symbol_info (symbol, filter_key, filter_value) values (%s, %s, %s)'
                cur.execute(qry, (symbol, ftype, json.dumps(fval)))

In [ ]:
[x for x in infos if x['symbol'] == '1INCHDOWNUSDT']

In [ ]:
from statsmodels.tsa.stattools import pacf
def get_adjusted_pcaf(x, nlags):
    ret = x.diff(1)/x.shift(1)
    ret = ret/ret.std()
    ret.loc[ret.abs() > 5] = np.nan
    ret = ret.dropna()
    if len(ret) < nlags*2:
        return
    val, cint = pacf(ret, nlags=nlags, alpha=0.05)
    adj = val[1:]/(cint[1:,1]-cint[1:,0])*2
    return adj
cdata = []
cols =  perpetual_6h.columns
start_date = '1-Jan-2021'
for c in cols:
    aa=perpetual_6h.loc[start_date:, c]
    nlags = 100
    ser = pd.Series(get_adjusted_pcaf(aa, nlags), index=list(range(1, nlags+1)))
    ser.name = c
    cdata.append(ser)
cdata = pd.concat(cdata, axis=1)
plt.axhspan(-1, 1, alpha=0.5, color='y')
cdata.mean(axis=1).plot(marker='o')
plt.grid(True)

In [ ]:
import time
def get_klines(symbol, starting):
    data = client.get_historical_klines(symbol=symbol, interval=client.KLINE_INTERVAL_1HOUR, start_str=starting)
    ser = pd.Series([vv[4] for vv in data], dtype=float, index=pd.to_datetime([vv[6] for vv in data], unit='ms'))
    ser.name = symbol
    return ser

symbols = client.get_all_tickers()
symbols = [x['symbol'] for x in symbols]
symbols = [x for x in symbols if x.endswith('USDT')]
starting = int(datetime.datetime.strptime('1-May-2021', '%d-%b-%Y').timestamp()*1000)
cols = []
for i, symbol in enumerate(symbols):
    curr_col = get_klines(symbol, starting)
    print(i, len(symbols), len(curr_col))
    cols.append(curr_col)
    time.sleep(0.01)
down = pd.concat(cols, axis=1)
#down.index = [x.replace(second=0, minute=0, microsecond=0) for x in down.index]
down

In [ ]:
with psycopg2.connect('host=localhost user=postgres password=YOUR_DB_PASSWORD port=5432') as conn:
    conn.autocommit = True
    with conn.cursor() as cur:
        print(idx)
        time_str = '10-Jun-2021 18:00'
        qry = 'insert into market (time, symbol, close) values(%s, %s, %s)'
        for symbol, val in down.iloc[2].iteritems():
            cur.execute(qry, (time_str, symbol, val))

In [ ]:
historical_live['ETHUSDT'].iloc[-5:]

In [ ]:
down.iloc[2].iteritems()

In [ ]:
pd.to_datetime(client.get_server_time()['serverTime'], unit='ms')

In [ ]:
with psycopg2.connect('host=localhost user=postgres password=YOUR_DB_PASSWORD port=5432') as conn:
    conn.autocommit = True
    with conn.cursor() as cur:
        for idx, row in down.iterrows():
            print(idx)
            time_str = idx.strftime('%d-%b-%Y %H:%M')
            qry = 'insert into market (time, symbol, close) values(%s, %s, %s)'
            for symbol, val in row.iteritems():
                try:
                    cur.execute(qry, (time_str, symbol, val))
                except UniqueViolation:
                    cur.execute('update market set close=%s where symbol=%s and time=%s', (val, symbol, time_str))


In [ ]:
from psycopg2.errors import UniqueViolation

In [ ]:
sec = get_secret('binance/binance/account2')
clienta = Client(sec['api_key'], sec['api_secret'])

In [ ]:
acc = clienta.get_account()
prices = {x['symbol']: float(x['price']) for x in clienta.get_all_tickers()}

In [ ]:
dust = [x for x in acc['balances'] if x['asset'] not in ['BNB', 'USDT'] and x['asset']+'USDT' in prices and 0. < float(x['free'])*prices[x['asset']+'USDT']<1.]
dust

In [ ]:
for dd in dust:
    try:
        clienta.transfer_dust(asset=dd['asset'])
        time.sleep(0.2)
    except Exception as ex:
        print(ex)

In [ ]:
def get_klines_futures(symbol):
    data = client.futures_klines(symbol=symbol, interval=client.KLINE_INTERVAL_1HOUR, start_str='1-Jan-2021')
    ser = pd.Series([vv[4] for vv in data], dtype=float, index=pd.to_datetime([vv[6] for vv in data], unit='ms'))
    ser.name = symbol
    return ser

In [ ]:
tickers = client.futures_symbol_ticker()
perpetual = []
for i, ticker in enumerate(tickers):
    print(i, len(tickers))
    perpetual.append(get_klines_futures(ticker['symbol']))
    time.sleep(0.1)
perpetual = pd.concat(perpetual, axis=1)
perpetual

In [ ]:
perpetual_1h = perpetual.copy()

In [ ]:
client.get_ticker(symbol='GTOUSDT')

In [ ]:
[x for x in client.get_all_tickers() if x['symbol'] == 'GTOUSDT']

In [ ]:
pd.to_datetime(1623620338110, unit='ms')

In [ ]:
client.get_orderbook_ticker(symbol='GTOUSDT')

In [ ]:
client.get_all_tickers()

In [ ]:
client.get_orderbook_tickers()

In [ ]:
[x for x in clienta.get_account()['balances'] if x['asset'] == 'ETHDOWN']

In [ ]:
clienta.redeem_lending_product(productId='ETHDOWNUSDT', type='FAST', amount=0.00132000)